In [ ]:
import os, glob, gc, re
import numpy as np
import xarray as xr
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# ✅ 导入计算函数
try:
    from aostools_functions import ComputeEPfluxDiv
except ImportError:
    import sys
    sys.path.insert(0, "/mnt/data")
    from aostools_functions import ComputeEPfluxDiv

# ============================================================
# 路径配置: 锁定到你的 BWCN 目录
# ============================================================
ROOT_DIR = "/mnt/soclim0/public_data/weiji/BWCN"
U_DIR     = os.path.join(ROOT_DIR, "U")
V_DIR     = os.path.join(ROOT_DIR, "V")
T_DIR     = os.path.join(ROOT_DIR, "T")
OMEGA_DIR = os.path.join(ROOT_DIR, "OMEGA")

OUT_DIR     = os.path.join(ROOT_DIR, "EPflux_daily")
OUT_FILE_NC = os.path.join(OUT_DIR, "EPFLUX_BWCN_Daily_24yr.nc")

os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- Settings ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=float)

LAT_RANGE   = (40.0, 80.0)
DO_UBAR     = False
WAVE        = -1
MAX_WORKERS = 24  # 对应 24 年数据，每核一年最快

# ---------------- Helpers ----------------
def compute_pressure_mid(ds: xr.Dataset) -> xr.DataArray:
    return ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]

def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, float)
    def _interp_col(vcol, pcol):
        m = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if m.sum() < 2: return np.full(p_tgt_pa.shape, np.nan, float)
        p_use, v_use = pcol[m], vcol[m]
        idx = np.argsort(p_use)
        return np.interp(np.log(p_tgt_pa), np.log(p_use[idx]), v_use[idx], left=np.nan, right=np.nan)

    dask_gufunc_kwargs = {"output_sizes": {"plev": len(p_tgt_pa)}}
    out = xr.apply_ufunc(
        _interp_col, v_hyb, p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True, dask="parallelized", output_dtypes=[float],
        dask_gufunc_kwargs=dask_gufunc_kwargs
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))

def sel_latband(da, lat0=40., lat1=80., lat_name="lat"):
    lat = da[lat_name]
    dec = float(lat.values[0]) > float(lat.values[-1])
    slc = slice(lat1, lat0) if dec else slice(lat0, lat1)
    return da.sel({lat_name: slc})

def coslat_weighted_mean(da, lat_name="lat"):
    lat = da[lat_name]
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(lat_name, skipna=True)

# ---------------- Worker ----------------
def process_one_year(year_str):
    """
    针对 24 年数据的简单逻辑，无需 time shift
    """
    try:
        # 文件命名匹配: BWCN.cam.h3.YYYY.U.nc
        fU = os.path.join(U_DIR, f"BWCN.cam.h3.{year_str}.U.nc")
        fV = os.path.join(V_DIR, f"BWCN.cam.h3.{year_str}.V.nc")
        fT = os.path.join(T_DIR, f"BWCN.cam.h3.{year_str}.T.nc")
        fW = os.path.join(OMEGA_DIR, f"BWCN.cam.h3.{year_str}.OMEGA.nc")

        for fp in [fU, fV, fT, fW]:
            if not os.path.exists(fp): return f"⚠️ Missing: {fp}"

        with xr.open_dataset(fU) as dsU, xr.open_dataset(fV) as dsV, \
             xr.open_dataset(fT) as dsT, xr.open_dataset(fW) as dsW:

            if dsU.sizes["time"] < 2: return None

            # 1) 计算气压层并插值
            p_mid = compute_pressure_mid(dsU)
            u_std = interp_profile_logp_4d(dsU["U"], p_mid, PLEV_STD_PA)
            v_std = interp_profile_logp_4d(dsV["V"], p_mid, PLEV_STD_PA)
            t_std = interp_profile_logp_4d(dsT["T"], p_mid, PLEV_STD_PA)
            w_std = interp_profile_logp_4d(dsW["OMEGA"] / 100.0, p_mid, PLEV_STD_PA)

            # 2) 提取 numpy 数据
            u_np, v_np, t_np, w_np = [d.transpose("time", "plev", "lat", "lon").values for d in [u_std, v_std, t_std, w_std]]
            lat_np = dsU["lat"].values

            # 3) 计算 EP Flux
            _, ep2, _, _ = ComputeEPfluxDiv(
                lat=lat_np, pres=(PLEV_STD_PA/100.0), 
                u=u_np, v=v_np, t=t_np, w=w_np, 
                do_ubar=DO_UBAR, wave=WAVE
            )

            # 4) 封装与空间平均
            da_ep2 = xr.DataArray(ep2, coords={"time": dsU["time"], "plev": PLEV_STD_PA, "lat": lat_np}, dims=("time", "plev", "lat"), name="Fz")
            out = coslat_weighted_mean(sel_latband(da_ep2, LAT_RANGE[0], LAT_RANGE[1], "lat")).compute()
            out.name = "Fz"

        gc.collect()
        return out
    except Exception as e:
        return f"Error in {year_str}: {str(e)}"

# ---------------- Main ----------------
def main():
    u_files = sorted(glob.glob(os.path.join(U_DIR, "BWCN.cam.h3.*.U.nc")))
    years = [os.path.basename(f).split('.')[3] for f in u_files] # 提取 YYYY
    
    if not years:
        print(f"❌ No files found in {U_DIR}")
        return

    print(f"🚀 BWCN EPflux (24yr): Processing {len(years)} years...")
    
    pool_results = []
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(process_one_year, y): y for y in years}
        for future in tqdm(as_completed(futures), total=len(futures)):
            res = future.result()
            if isinstance(res, xr.DataArray): pool_results.append(res)
            elif res: print(res)

    if pool_results:
        Fz_all = xr.concat(pool_results, dim="time").sortby("time")
        ds_out = xr.Dataset({"Fz": Fz_all})
        ds_out.attrs = {"description": "BWCN EP Flux (Fz) 40-80N Daily mean", "units": "m^2/s^2"}
        ds_out.to_netcdf(OUT_FILE_NC)
        print(f"✅ BWCN Done: {OUT_FILE_NC}")

if __name__ == "__main__":
    main()